In [1]:
from pydrake.all import (
    StartMeshcat,
    AddMultibodyPlantSceneGraph,
    Parser,
    AddDefaultVisualization,
    DiagramBuilder,
    PointCloud,
    Fields,
    BaseField,
    RigidTransform,
    RotationMatrix,
    InverseKinematics,
    Solve,
)
from pathlib import Path
import numpy as np
import open3d as o3d
from scipy.spatial.transform import Rotation

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## Setup

In [2]:
meshcat = StartMeshcat()

INFO:drake:Meshcat listening for connections at http://localhost:7000


In [2]:
project_dir = Path("/home/noor/so101-drake")

In [4]:
builder = DiagramBuilder()

plant, scene_graph = AddMultibodyPlantSceneGraph(
    builder,
    time_step=0.001,
)

parser = Parser(plant)
parser.AddModels(project_dir / "models" / "objects" / "surface.sdf")
plant.WeldFrames(
    plant.world_frame(),
    plant.GetFrameByName("surface_link")
)

T_world_base = np.eye(4)
rotvec = np.array([0, 0, 1]) * np.pi/2
T_world_base[:3, :3] = Rotation.from_rotvec(rotvec).as_matrix()
T_world_base[:3, 3] = [0, -0.1775, 0.0074]

so101 = parser.AddModels(
    project_dir / "models" / "SO101" / "so101_new_calib_drake_hydro.urdf"
)[0]
plant.WeldFrames(
    plant.world_frame(),
    plant.GetFrameByName("base_link"),
    RigidTransform(
        RotationMatrix(T_world_base[:3, :3]), 
        T_world_base[:3, 3]
    )
)

# grasp = np.array([
#     [ 0.20856473, -0.96235112, -0.17430128, -0.04784185],
#     [-0.97346751, -0.22142707,  0.05771393,  0.04177497],
#     [-0.09413604,  0.15763952, -0.98299962,  0.15351727],
#     [ 0.        ,  0.        ,  0.        ,  1.        ],
# ])
# grasp = np.eye(4)
# grasp[:3, :3] = Rotation.from_rotvec([0, 0, np.pi/2]).as_matrix()
# grasp[:3, 3] = [0.13, 0.02, 0.2]

# g = parser.AddModels("/home/noor/SO-ARM100/Simulation/SO101/so101_gripper.urdf")[0]
# plant.WeldFrames(
#     plant.world_frame(), 
#     plant.GetFrameByName("gripper_link", g), 
#     RigidTransform(
#         # RotationMatrix.MakeYRotation(np.pi) @ RotationMatrix(grasp[:3, :3]),
#         RotationMatrix(grasp[:3, :3]),
#         # [grasp[0, 3], grasp[1, 3]+0.015, grasp[2, 3]]
#         grasp[:3, 3]
#     )
# )

plant.Finalize()

AddDefaultVisualization(builder, meshcat)

diagram = builder.Build()
context = diagram.CreateDefaultContext()

q_init = np.zeros(6)
q_rest = np.array([0, -1.822, 1.55, 0.906, 0, 0])
plant.SetPositions(plant.GetMyMutableContextFromRoot(context), q_rest)

diagram.ForcedPublish(context)

==== LCM Warning ===
LCM detected that large packets are being received, but the kernel UDP
receive buffer is very small.  The possibility of dropping packets due to
insufficient buffer space is very high.

For more information, visit:
   https://lcm-proj.github.io/lcm/content/multicast-setup.html



In [5]:
filtered = o3d.io.read_point_cloud(project_dir / "assets" / "filtered.ply")

points = np.asarray(filtered.points)
colors = np.asarray(filtered.colors)  # sRGB in [0, 1] from Open3D

def srgb_to_linear(c):
    c = np.clip(c, 0.0, 1.0)
    return np.where(c <= 0.04045, c / 12.92, ((c + 0.055) / 1.055) ** 2.4)

colors_linear = srgb_to_linear(colors)

# Round and clip before casting to avoid uint8 wraparound.
rgbs_uint8 = np.clip(np.round(255.0 * colors_linear), 0, 255).astype(np.uint8)
rgbs_uint8 = np.flip(rgbs_uint8, axis=1)

pc = PointCloud(
    new_size=len(points),
    fields=Fields(BaseField.kXYZs | BaseField.kRGBs),
)
pc.mutable_xyzs()[:] = points.T
pc.mutable_rgbs()[:] = rgbs_uint8.T

meshcat.SetObject("/my_surface/rgb_points", pc, point_size=0.003)

## Inverse Kinematics

In [7]:
def solve_ik_place():
    grasp = np.eye(4)
    grasp[:3, :3] = Rotation.from_rotvec([0, 0, np.pi/2]).as_matrix()
    grasp[:3, 3] = [0.13, 0.0, 0.2]

    # initial guess
    ik = InverseKinematics(plant)
    ik.get_mutable_prog().AddQuadraticErrorCost(1.0, q_init, ik.q())
    ik.get_mutable_prog().SetInitialGuess(ik.q(), q_init)

    # constraints
    ik.AddPositionConstraint(
        plant.GetFrameByName("gripper_link", so101),
        [0.0, 0.0, 0.0],
        plant.world_frame(),
        grasp[:3, 3],
        grasp[:3, 3]
    )
    ik.AddOrientationConstraint(
        plant.GetFrameByName("gripper_link", so101),
        RotationMatrix(),
        plant.world_frame(),
        RotationMatrix(grasp[:3, :3]),
        np.pi/16
    )
    ik.get_mutable_prog().AddBoundingBoxConstraint(
        np.pi/4, np.pi/4, ik.q()[5]
    )

    # solve
    result = Solve(ik.prog())
    if result.is_success():
        plant.SetPositions(
            plant.GetMyMutableContextFromRoot(context), 
            result.GetSolution(ik.q())
        )
        diagram.ForcedPublish(context)
        return result.GetSolution(ik.q())
    else:
        return None


q_place = solve_ik_place()
print(q_place)

[ 0.75222731 -0.1107525  -0.007926    1.50925026  0.72288082  0.78539816]


In [8]:
def solve_ik_pick(grasp):
    # initial guess
    ik = InverseKinematics(plant)
    ik.get_mutable_prog().AddQuadraticErrorCost(1.0, q_init, ik.q())
    ik.get_mutable_prog().SetInitialGuess(ik.q(), q_init)

    # constraints
    ik.AddPositionConstraint(
        plant.GetFrameByName("gripper_link", so101),
        [0.0, 0.0, 0.0],
        plant.world_frame(),
        grasp[:3, 3] + np.array([-0.015, 0.01, 0]),
        grasp[:3, 3] + np.array([-0.015, 0.01, 0])
    )
    ik.AddOrientationConstraint(
        plant.GetFrameByName("gripper_link", so101),
        RotationMatrix(),
        plant.world_frame(),
        RotationMatrix.MakeYRotation(np.pi) @ RotationMatrix(grasp[:3, :3]),
        np.pi/16
    )
    ik.get_mutable_prog().AddBoundingBoxConstraint(
        np.pi/4, np.pi/4, ik.q()[5]
    )

    # solve
    result = Solve(ik.prog())
    if result.is_success():
        plant.SetPositions(
            plant.GetMyMutableContextFromRoot(context), 
            result.GetSolution(ik.q())
        )
        diagram.ForcedPublish(context)
        return result.GetSolution(ik.q())
    else:
        return None


grasp = np.array([
    [ 0.20856473, -0.96235112, -0.17430128, -0.04784185],
    [-0.97346751, -0.22142707,  0.05771393,  0.04177497],
    [-0.09413604,  0.15763952, -0.98299962,  0.15351727],
    [ 0.        ,  0.        ,  0.        ,  1.        ],
])
q_pick = solve_ik_pick(grasp)
print(q_pick)

[-0.31961414  0.05712364  0.13169853  1.34462259  2.59307222  0.78539816]


## Motion Planning

In [11]:
meshcat = StartMeshcat()

INFO:drake:Meshcat listening for connections at http://localhost:7001


In [4]:
import coacd
import trimesh


mesh = trimesh.load("/home/noor/so101-drake/assets/mesh.obj", force="mesh")
mesh = coacd.Mesh(mesh.vertices, mesh.faces)
parts = coacd.run_coacd(mesh, threshold=0.01, real_metric=True)

[2026-09-02 21:10:57.604] [CoACD] [info] threshold               0.01
[2026-09-02 21:10:57.604] [CoACD] [info] max # convex hull       -1
[2026-09-02 21:10:57.604] [CoACD] [info] preprocess mode         auto
[2026-09-02 21:10:57.604] [CoACD] [info] preprocess resolution   50
[2026-09-02 21:10:57.604] [CoACD] [info] pca                     false
[2026-09-02 21:10:57.604] [CoACD] [info] mcts max depth          3
[2026-09-02 21:10:57.604] [CoACD] [info] mcts nodes              20
[2026-09-02 21:10:57.604] [CoACD] [info] mcts iterations         150
[2026-09-02 21:10:57.604] [CoACD] [info] merge                   true
[2026-09-02 21:10:57.604] [CoACD] [info] decimate                false
[2026-09-02 21:10:57.604] [CoACD] [info] max_ch_vertex           256
[2026-09-02 21:10:57.604] [CoACD] [info] extrude                 false
[2026-09-02 21:10:57.604] [CoACD] [info] extrude margin          0.01
[2026-09-02 21:10:57.604] [CoACD] [info] approximate mode        ch
[2026-09-02 21:10:57.604] [CoA

In [12]:
from pydrake.all import (
    RobotDiagramBuilder,
    RobotDiagram,
    Convex,
    InMemoryMesh,
    MemoryFile,
    CoulombFriction,
    MeshcatVisualizer,
    MeshcatVisualizerParams,
    Role,
)

robot_builder = RobotDiagramBuilder()
builder = robot_builder.builder()
plant = robot_builder.plant()
scene_graph = robot_builder.scene_graph()

parser = Parser(plant)

print("starting mesh registration...")
for i, (v, f) in enumerate(parts):
    obj_str = trimesh.Trimesh(v, f).export(file_type="obj")
    shape = Convex(InMemoryMesh(mesh_file=MemoryFile(obj_str, ".obj", f"part_{i}.obj")))
    plant.RegisterCollisionGeometry(
        plant.world_body(), RigidTransform(), shape,
        f"scene_part_{i}", CoulombFriction(1.0, 1.0))
print("finished mesh registration")    

T_world_base = np.eye(4)
rotvec = np.array([0, 0, 1]) * np.pi/2
T_world_base[:3, :3] = Rotation.from_rotvec(rotvec).as_matrix()
T_world_base[:3, 3] = [0, -0.1775, 0.0074]

so101 = parser.AddModels(
    project_dir / "models" / "SO101" / "so101_new_calib_drake_hydro.urdf"
)[0]
plant.WeldFrames(
    plant.world_frame(),
    plant.GetFrameByName("base_link"),
    RigidTransform(
        RotationMatrix(T_world_base[:3, :3]), 
        T_world_base[:3, 3]
    )
)

plant.Finalize()

# AddDefaultVisualization(builder, meshcat)
visualizer = MeshcatVisualizer.AddToBuilder(
    builder,
    scene_graph,
    meshcat,
    MeshcatVisualizerParams(role=Role.kIllustration),
)
collision_visualizer = MeshcatVisualizer.AddToBuilder(
    builder,
    scene_graph,
    meshcat,
    MeshcatVisualizerParams(
        prefix="collision", role=Role.kProximity, visible_by_default=False
    ),
)

diagram: RobotDiagram = robot_builder.Build()
context = diagram.CreateDefaultContext()

q_init = np.zeros(6)
q_rest = np.array([0, -1.822, 1.55, 0.906, 0, 0])
plant.SetPositions(plant.GetMyMutableContextFromRoot(context), q_rest)

diagram.ForcedPublish(context)

filtered = o3d.io.read_point_cloud(project_dir / "assets" / "filtered.ply")

points = np.asarray(filtered.points)
colors = np.asarray(filtered.colors)  # sRGB in [0, 1] from Open3D

def srgb_to_linear(c):
    c = np.clip(c, 0.0, 1.0)
    return np.where(c <= 0.04045, c / 12.92, ((c + 0.055) / 1.055) ** 2.4)

colors_linear = srgb_to_linear(colors)

# Round and clip before casting to avoid uint8 wraparound.
rgbs_uint8 = np.clip(np.round(255.0 * colors_linear), 0, 255).astype(np.uint8)
rgbs_uint8 = np.flip(rgbs_uint8, axis=1)

pc = PointCloud(
    new_size=len(points),
    fields=Fields(BaseField.kXYZs | BaseField.kRGBs),
)
pc.mutable_xyzs()[:] = points.T
pc.mutable_rgbs()[:] = rgbs_uint8.T

meshcat.SetObject("/my_surface/rgb_points", pc, point_size=0.003)

starting mesh registration...
finished mesh registration


In [13]:
q_place = np.array([
    0.75222731, -0.1107525, -0.007926, 1.50925026, 0.72288082, 0.78539816
])
q_pick = np.array([
    -0.31961414, 0.05712364, 0.13169853, 1.34462259, 2.59307222, 0.78539816
])

plant.SetPositions(
    plant.GetMyMutableContextFromRoot(context), 
    q_pick
)
diagram.ForcedPublish(context)

In [14]:
from ompl_planning import SO101SamplingPlanner


waypoints = SO101SamplingPlanner.generate_path(plant, diagram, q_pick, q_place)
waypoints.shape

INFO:drake:Allocating contexts to support implicit context parallelism 24


6
Debug:   RRTConnectIntermediate: Planner range detected to be 20.095896
Info:    RRTConnectIntermediate: Starting planning with 1 states already in datastructure
Info:    RRTConnectIntermediate: Created 15 states (8 start + 7 goal)


(6, 14)

In [15]:
waypoints_trunc = np.hstack((waypoints[:, :5], waypoints[:, -3:]))
waypoints_trunc.shape

(6, 8)

In [16]:
from pydrake.all import (
    PiecewisePolynomial,
)
from manipulation.meshcat_utils import PublishPositionTrajectory


times = np.linspace(0.0, 2.0, waypoints_trunc.shape[1])
ompl_traj = PiecewisePolynomial.FirstOrderHold(times, waypoints_trunc)

meshcat.Flush()
context = diagram.CreateDefaultContext()
PublishPositionTrajectory(ompl_traj, context, plant, visualizer)
collision_visualizer.ForcedPublish(collision_visualizer.GetMyContextFromRoot(context))


In [17]:
from pydrake.all import (
    MultibodyPlant,
    ModelInstanceIndex,
    BsplineTrajectory,
    BsplineBasis,
    KinematicTrajectoryOptimization,
    CollisionCheckerParams,
    SceneGraphCollisionChecker,
    MinimumDistanceLowerBoundConstraint,
)


def GenerateTrajectory(
    plant: MultibodyPlant,
    diagram: RobotDiagram,
    robot_model: ModelInstanceIndex,
    waypoints: np.ndarray,
    avoid_collisions = True,
) -> BsplineTrajectory | None:
    num_q = plant.num_positions()
    basis = BsplineBasis(4, waypoints.shape[1])
    init_traj = BsplineTrajectory(basis, waypoints)
    trajopt = KinematicTrajectoryOptimization(num_q, waypoints.shape[1], 4)
    trajopt.SetInitialGuess(init_traj)

    trajopt.AddDurationCost(1.0)
    trajopt.AddPathLengthCost(1.0)
    trajopt.AddPositionBounds(
        plant.GetPositionLowerLimits(), 
        plant.GetPositionUpperLimits()
    )
    trajopt.AddVelocityBounds(
        0.3 * plant.GetVelocityLowerLimits(),
        0.3 * plant.GetVelocityUpperLimits()
    )
    trajopt.AddAccelerationBounds(
        -2.0 * np.ones(num_q),
         2.0 * np.ones(num_q)
    )
    trajopt.AddJerkBounds(
        -1.0 * np.ones(num_q),
         1.0 * np.ones(num_q)
    )
    trajopt.AddDurationConstraint(0.5, 5.0)

    q_start = waypoints[:, 0]
    q_goal = waypoints[:, -1]
    trajopt.AddPathPositionConstraint(lb=q_start, ub=q_start, s=0)
    trajopt.AddPathPositionConstraint(lb=q_goal, ub=q_goal, s=1)

    trajopt.AddPathVelocityConstraint(np.zeros((num_q, 1)), np.zeros((num_q, 1)), 0)
    trajopt.AddPathVelocityConstraint(np.zeros((num_q, 1)), np.zeros((num_q, 1)), 1)

    if avoid_collisions:
        collision_checker_params = CollisionCheckerParams()
        collision_checker_params.model = diagram
        collision_checker_params.robot_model_instances = [robot_model]
        collision_checker_params.edge_step_size = 0.01
        collision_checker = SceneGraphCollisionChecker(collision_checker_params)
        # collision_checker.SetPaddingAllRobotEnvironmentPairs(1e-3)
        collision_constraint = MinimumDistanceLowerBoundConstraint(
            collision_checker,
            5e-3,
            collision_checker.MakeStandaloneModelContext(),
            None,
            5e-2,
        )
        evaluate_at_s = np.linspace(0, 1, 25)
        for s in evaluate_at_s:
            trajopt.AddPathPositionConstraint(collision_constraint, s)

    prog = trajopt.get_mutable_prog()
    result = Solve(prog)
    if result.is_success():
        return trajopt.ReconstructTrajectory(result)
    else:
        return None


trajectory = GenerateTrajectory(plant, diagram, so101, waypoints_trunc)
print(trajectory)

INFO:drake:Allocating contexts to support implicit context parallelism 24


In [18]:
meshcat.Flush()
context = diagram.CreateDefaultContext()
PublishPositionTrajectory(trajectory, context, plant, visualizer)
collision_visualizer.ForcedPublish(collision_visualizer.GetMyContextFromRoot(context))